# Churn (lapse) EDA — eudirectlapse

Exploratory analysis of the public `eudirectlapse` dataset (CASdatasets, GPL >= 2):
23,060 European direct motor-insurance renewals, binary `lapse` target.

**Important framing.** This is a *European direct insurer* with an overall lapse of
~12.8%. We do **not** use its absolute level for the UK — the UK motor base churn
stays pinned to the FCA figure (~40%, see `docs/calibration.md`). This dataset is
used only for:

1. **Relative risk** between customers (how lapse varies by age, tenure, number of
   policies). We rescale model predictions so the portfolio mean equals the FCA base.
2. **The price-change effect** (lapse at the new price minus the old price).

Run `python scripts/download_data.py` first — this notebook reads `data/raw/`,
which is git-ignored.

In [ ]:
import numpy as np
import pandas as pd

from decision_sim.ml.data import prepare

df = prepare("../data/raw/eudirectlapse.parquet")
print(f"rows: {len(df):,}")
print(f"overall lapse rate: {df['lapse'].mean():.4f}")
df.head()

## 1. Lapse by driver age

Clear, monotonic signal: younger drivers lapse much more than older ones
(~17% under 30 vs ~9% for 60+). This is a genuine, usable pattern.

In [ ]:
age_bins = pd.cut(df["polholder_age"], bins=[18, 30, 45, 60, 100],
                  labels=["<30", "30-44", "45-59", "60+"])
df.groupby(age_bins, observed=True)["lapse"].agg(["mean", "size"]).round(4)

## 2. Lapse by policy tenure (years with the insurer)

Longer-tenured customers are stickier: ~8% lapse at 6+ years vs ~14-15% in the
first couple of years. Also a clean pattern.

In [ ]:
tenure_bins = pd.cut(df["policy_age"], bins=[-1, 0, 2, 5, 100],
                     labels=["0", "1-2", "3-5", "6+"])
df.groupby(tenure_bins, observed=True)["lapse"].agg(["mean", "size"]).round(4)

## 3. Lapse by number of policies held

Weaker and noisier (small cells at high counts), but broadly customers with more
contracts are no more likely to lapse — mild stickiness.

In [ ]:
df.groupby("policy_nbcontract", observed=True)["lapse"].agg(["mean", "size"]).round(4)

## 4. The price-change confounding (read this carefully)

Intuition says: raise the price → more people leave. The raw data shows the
**opposite**:

- Customers whose price was **cut** lapse the most (~16%).
- Customers given a **10%+ increase** lapse the *least* (~10%).
- Correlation of `price_change` with `lapse` is slightly **negative** (~-0.02).

This is **confounding**, not a real "higher price → less churn" effect: insurers
give the biggest increases to the customers they already know are sticky (loyal,
low-risk, unlikely to shop around), and offer discounts to those at risk of
leaving. The price variable is entangled with the retention decision.

**Consequence for our model.** We cannot read a clean causal price elasticity off
this raw data. So:

- We keep the labelled `max_extra_churn` **assumption** (0.15) from Phase 7 as the
  fallback and the **upper bound** for extra churn caused by a price rise.
- The model's price-change delta (pred at new price − pred at old price) is used
  only if it is positive and plausible; otherwise we fall back to the assumption.
- This is reported honestly in the UI and `docs/calibration.md` rather than
  pretending the model found a strong price effect.

In [ ]:
price_bins = pd.cut(df["price_change"], bins=[-10, -0.001, 0.001, 0.05, 0.10, 10],
                    labels=["cut", "flat", "0-5%", "5-10%", "10%+"])
summary = df.groupby(price_bins, observed=True)["lapse"].agg(["mean", "size"]).round(4)
print(summary)
print(f"\ncorr(price_change, lapse)   = {df['price_change'].corr(df['lapse']):.4f}")
print(f"corr(price_vs_market, lapse) = {df['price_vs_market'].corr(df['lapse']):.4f}")

## 5. Lapse by price vs market

`price_vs_market = prem_final / prem_market - 1`. Being priced above the market
nudges lapse up a little (0-10% above market ~14% vs well below market ~12%), but
the signal is weak and also tangled with who gets which price.

In [ ]:
mkt_bins = pd.cut(df["price_vs_market"], bins=[-10, -0.1, 0, 0.1, 10],
                  labels=["<-10%", "-10-0%", "0-10%", "10%+"])
df.groupby(mkt_bins, observed=True)["lapse"].agg(["mean", "size"]).round(4)

## 6. Model performance

Trained via `scripts/train_churn.py` (LogisticRegression baseline +
HistGradientBoostingClassifier). On a 25% stratified hold-out (seed 7):

| Model | ROC AUC | Brier |
|---|---|---|
| LogReg | 0.624 | 0.110 |
| HGB | 0.620 | 0.109 |

Modest discrimination — churn here is genuinely hard to predict from these
features, and the price signal is confounded. That is expected and reported
honestly. The value we take from the model is the **relative ordering** of
customers (age/tenure patterns above), which we rescale to the FCA base in
`decision_sim.ml.predict` (Phase 8d).

In [ ]:
from decision_sim.ml.data import prepare
from decision_sim.ml.train import train_model

features = prepare("../data/raw/eudirectlapse.parquet")
for name in ("logreg", "hgb"):
    r = train_model(features, name)
    print(f"{name:>7}: ROC AUC {r.roc_auc:.4f} | Brier {r.brier:.4f}")